# TED semantic base exploration

Read the **first column** of `List_of_search_fields.csv` and recursively group its hyphen-separated names. Only Python's standard library is needed.

1. Group distinct names by their first word, retaining distinct remaining suffixes. A name ending at the current key contributes `None`.
2. Move common leading words from **all** suffixes into their key, stopping at a branch or a terminal suffix. Empty suffixes become `None`.
3. Repeat for each suffix group until every leaf is `[None]`, returning a list of single-key maps sorted by their compressed keys at every level.

`None` is Python's terminal marker, not the string `"none"`. When a name is also a prefix of longer names, a bare `None` list element preserves that name. For example, `['buyer', 'buyer-name']` becomes `[{'buyer': [None, {'name': [None]}]}]`. A single path such as `['a-b-c']` compresses to `[{'a-b-c': [None]}]`. Empty input produces `[]`. Duplicate names are removed; spelling and case are preserved. Maps use Python's case-sensitive alphabetical ordering by key, with the terminal `None` before child maps.

This is a lexical hierarchy of TED search-field names; shared words do not establish formal ontology relationships. The SDK metadata join reuses [semantic_base_builder.py](semantic_base_builder.py).

Each top-level entry also contains `_N_CHILDREN`, the number of distinct complete fields represented by that branch, including the key itself when it is a field. For example, `['alfa', 'alfa-beta', 'alfa-beta-gamma']` becomes `[{'alfa': [None, {'beta': [None, {'gamma': [None]}]}], '_N_CHILDREN': 3}]`. Nested entries retain their single field key. `_N_CHILDREN` is reserved for metadata and has no trailing space. `restore_names` accepts both plain and enriched prefix entries, ignores metadata when finding the branch, and verifies any supplied count. The flat suffix mapper uses its separate reader, `restore_suffix_names`.

In [2]:
import csv
from pathlib import Path
from pprint import pprint

# Locate the input from the repository root, TED folder, or a descendant folder.
def find_search_fields(start):
    for folder in (start, *start.parents):
        for candidate in (
            folder / "List_of_search_fields.csv",
            folder / "API map" / "List_of_search_fields.csv",
            folder / "sources" / "TED" / "semantic base" / "API map" / "List_of_search_fields.csv",
        ):
            if candidate.is_file():
                return candidate
    raise FileNotFoundError("Cannot locate sources/TED/semantic base/API map/List_of_search_fields.csv")


input_path = find_search_fields(Path.cwd())
with input_path.open(encoding="utf-8-sig", newline="") as handle:
    rows = csv.reader(handle)
    header = next(rows)
    search_fields = [row[0].strip() for row in rows if row and row[0].strip()]

print(f"Column: {header[0]}; rows: {len(search_fields)}; distinct: {len(set(search_fields))}")

Column: Search field; rows: 1340; distinct: 1340


In [2]:
# Step 0: group each first word with its distinct following suffixes.
def first_level_map(names):
    names = list(dict.fromkeys(names))
    if names == [None]:
        return names

    groups = {}
    for name in names:
        if name is None:
            key, suffix = None, None
        else:
            key, separator, remainder = name.partition("-")
            suffix = remainder if separator and remainder else None
        groups.setdefault(key, {})[suffix] = None
    return {key: list(following) for key, following in groups.items()}


# Step 1: compress shared leading words without crossing a terminal suffix.
def compress_group(key, suffixes):
    while suffixes and None not in suffixes:
        parts = [suffix.partition("-") for suffix in suffixes]
        word = parts[0][0]
        if any(part[0] != word for part in parts):
            break
        key = f"{key}-{word}"
        suffixes = list(dict.fromkeys(
            remainder if separator and remainder else None
            for _, separator, remainder in parts
        ))
    return key, suffixes


# Step 2: recursively turn each compressed suffix group into the next level.
def _hierarchical_map(names):
    groups = first_level_map(names)
    if groups == [None]:
        return groups

    result = []
    for key, suffixes in groups.items():
        if key is None:
            result.append(None)
            continue
        key, suffixes = compress_group(key, suffixes)
        result.append({key: _hierarchical_map(suffixes)})
    # Sort after compression because the final keys determine sibling order.
    result.sort(key=lambda item: (
        item is not None, next(iter(item)) if item is not None else ""
    ))
    return result



# Read the field branch independently of metadata order in the dictionary.
def field_branch(item):
    assert isinstance(item, dict)
    branches = [(key, value) for key, value in item.items() if key != "_N_CHILDREN"]
    assert len(branches) == 1, "Each entry must have exactly one field branch"
    return branches[0]


def restore_names(node, prefix=()):
    assert isinstance(node, list), "Every level must be a list"
    branches = [None if item is None else field_branch(item) for item in node]
    keys = [None if branch is None else branch[0] for branch in branches]
    assert all(branch[0] is not None for branch in branches if branch is not None)
    assert len(keys) == len(set(keys)), "Sibling keys and terminal markers must be unique"
    assert keys == sorted(keys, key=lambda key: (key is not None, key or ""))
    names = []
    for item, branch in zip(node, branches):
        if item is None:
            names.append("-".join(prefix))
            continue
        key, child = branch
        restored = restore_names(child, (*prefix, key))
        if "_N_CHILDREN" in item:
            assert type(item["_N_CHILDREN"]) is int and item["_N_CHILDREN"] == len(restored)
        names.extend(restored)
    return names


# Enrich only top-level entries; count complete fields, not immediate branches.
def hierarchical_map(names):
    result = _hierarchical_map(names)
    for item in result:
        if item is not None:
            _, children = field_branch(item)
            item["_N_CHILDREN"] = len(restore_names(children))
    return result

## Check compression and terminal names

These examples cover a shared multiword prefix, a name that is also a parent, duplicates, an empty list, and the `[None]` stopping rule.

In [3]:
assert hierarchical_map([None]) == [None]
assert hierarchical_map([]) == []
assert hierarchical_map(["a-b-d", "a-b-c", "a-b-c"]) == [
    {"a-b": [{"c": [None]}, {"d": [None]}], "_N_CHILDREN": 2}
]
assert hierarchical_map(["z-q-r", "a-b-c", "a-b", "a"]) == [
    {"a": [None, {"b": [None, {"c": [None]}]}], "_N_CHILDREN": 3},
    {"z-q-r": [None], "_N_CHILDREN": 1},
]
assert hierarchical_map(["a-bc", "a-b"]) == [
    {"a": [{"b": [None]}, {"bc": [None]}], "_N_CHILDREN": 2}
]
assert hierarchical_map(["a-z", "a!-x"]) == [
    {"a!-x": [None], "_N_CHILDREN": 1}, {"a-z": [None], "_N_CHILDREN": 1}
]
assert hierarchical_map(["alfa", "alfa-beta", "alfa-beta-gamma"]) == [
    {"alfa": [None, {"beta": [None, {"gamma": [None]}]}], "_N_CHILDREN": 3}
]
assert restore_names([{"_N_CHILDREN": 2, "alfa": [None, {"beta": [None]}]}]) == [
    "alfa", "alfa-beta"
]
print("Example checks passed.")

Example checks passed.


In [4]:
# Build the full hierarchy and verify that it preserves every distinct name.
mapping = hierarchical_map(search_fields)

In [5]:
restored = restore_names(mapping)
assert len(restored) == len(set(search_fields))
assert set(restored) == set(search_fields)
print(f"Verified {len(restored)} distinct fields; {len(mapping)} top-level groups.")
assert sum(item["_N_CHILDREN"] for item in mapping) == len(set(search_fields))

Verified 1340 distinct fields; 143 top-level groups.


In [6]:
# Display the complete hierarchy with alphabetically ordered sibling maps.
pprint(mapping, sort_dicts=False, width=100)

[{'AC': [None], '_N_CHILDREN': 1},
 {'FT': [None], '_N_CHILDREN': 1},
 {'HA': [None], '_N_CHILDREN': 1},
 {'NL': [None], '_N_CHILDREN': 1},
 {'RN': [None], '_N_CHILDREN': 1},
 {'RP': [None], '_N_CHILDREN': 1},
 {'TD': [None], '_N_CHILDREN': 1},
 {'TY': [None], '_N_CHILDREN': 1},
 {'accessibility': [{'justification-lot': [None]}, {'lot': [None]}], '_N_CHILDREN': 2},
 {'acquiring-cpb-buyer': [None], '_N_CHILDREN': 1},
 {'activity-sector': [None], '_N_CHILDREN': 1},
 {'additional': [{'classification': [{'lot': [None]},
                                     {'part': [None]},
                                     {'proc': [None]},
                                     {'type': [{'lot': [None]},
                                               {'part': [None]},
                                               {'proc': [None]}]}]},
                 {'info': [{'glo': [None]}, {'part': [None]}, {'proc': [None]}]},
                 {'information': [None, {'lot': [None]}]}],
  '_N_CHILDREN': 11},
 {'ann

In [7]:
# Rank top-level groups using their stored field counts.
k_list = [(field_branch(item)[0], item["_N_CHILDREN"]) for item in mapping]
k_list.sort(key=lambda item: item[1], reverse=True)
k_list

[('organisation', 420),
 ('touchpoint', 364),
 ('buyer', 32),
 ('place', 28),
 ('winner', 27),
 ('contract', 24),
 ('review', 23),
 ('framework', 18),
 ('award', 17),
 ('tender', 17),
 ('deadline', 14),
 ('business', 12),
 ('document', 12),
 ('subcontracting', 12),
 ('additional', 11),
 ('selection', 9),
 ('ubo', 9),
 ('electronic', 8),
 ('estimated-value', 8),
 ('change', 7),
 ('eu', 7),
 ('main', 7),
 ('notice', 7),
 ('result', 7),
 ('dispatch', 6),
 ('legal-basis', 6),
 ('modification', 6),
 ('previous', 6),
 ('url', 6),
 ('concession', 5),
 ('duration', 5),
 ('eed', 5),
 ('group-framework', 5),
 ('procedure', 5),
 ('reserved', 5),
 ('description', 4),
 ('energy', 4),
 ('internal-identifier', 4),
 ('procurement', 4),
 ('public-opening', 4),
 ('registration', 4),
 ('submission', 4),
 ('title', 4),
 ('tool', 4),
 ('direct-award-justification', 3),
 ('dps', 3),
 ('environmental', 3),
 ('exclusion-grounds', 3),
 ('identifier', 3),
 ('lots', 3),
 ('non', 3),
 ('publication', 3),
 ('sme',

## Shared suffix map

Group distinct field names by their last hyphen-separated token. Keep only groups containing **at least two distinct fields**, then factor their shared trailing tokens into the suffix key. Values are the remaining prefixes, sorted alphabetically with `None` first when the suffix itself is a field. The output is one level deep, with alphabetically sorted single-key maps.

For example, `['alfa-beta-lot', 'gamma-beta-lot', 'buyer-country']` produces `[{'beta-lot': ['alfa', 'gamma']}]`; `buyer-country` is excluded because its final-token group has only one distinct field. Duplicate rows do not increase a group's size.

This is a discovery view rather than a complete inventory. `excluded_suffix_fields` reports fields omitted from the suffix map. The recursive prefix hierarchy above still covers every field.

In [8]:
# Reverse token order, preserving the characters within each token.
def reverse_tokens(name):
    return "-".join(reversed(name.split("-")))


# Reuse prefix grouping and compression on reversed tokens, without recursion.
def suffix_map(names):
    groups = first_level_map(reverse_tokens(name) for name in dict.fromkeys(names))
    result = []
    for key, remainders in groups.items():
        if len(remainders) < 2:
            continue
        key, remainders = compress_group(key, remainders)
        suffix = reverse_tokens(key)
        prefixes = [None if remainder is None else reverse_tokens(remainder) for remainder in remainders]
        prefixes.sort(key=lambda prefix: (prefix is not None, prefix or ""))
        result.append({suffix: prefixes})
    return sorted(result, key=lambda item: next(iter(item)))


# Reconstruct retained names and verify the flat shape and alphabetical ordering.
def restore_suffix_names(groups):
    names = []
    keys = []
    for item in groups:
        assert isinstance(item, dict) and len(item) == 1
        suffix, prefixes = next(iter(item.items()))
        assert isinstance(suffix, str) and isinstance(prefixes, list)
        assert len(prefixes) >= 2, "Each suffix must occur in at least two distinct fields"
        assert all(prefix is None or isinstance(prefix, str) for prefix in prefixes)
        assert prefixes == sorted(set(prefixes), key=lambda prefix: (prefix is not None, prefix or ""))
        keys.append(suffix)
        names.extend(suffix if prefix is None else f"{prefix}-{suffix}" for prefix in prefixes)
    assert keys == sorted(set(keys))
    assert len(names) == len(set(names)), "Each retained field must occur exactly once"
    return names

In [9]:
# Check shared endings, deduplication, terminal names, and token boundaries.
assert suffix_map([]) == []
assert suffix_map(["buyer-country", "buyer-country"]) == []
assert suffix_map(["alfa-beta-lot", "gamma-beta-lot", "buyer-country"]) == [
    {"beta-lot": ["alfa", "gamma"]}
]
assert suffix_map(["lot", "classification-lot", "additional-classification-lot"]) == [
    {"lot": [None, "additional-classification", "classification"]}
]
assert suffix_map(["left-a-shared-end", "right-b-shared-end"]) == [
    {"shared-end": ["left-a", "right-b"]}
]
assert suffix_map(["a-tail", "b-retail"]) == []
assert suffix_map(["b-b-z", "a-b-z", "b-a-y", "a-a-y"]) == [
    {"a-y": ["a", "b"]}, {"b-z": ["a", "b"]}
]
print("Suffix example checks passed.")

Suffix example checks passed.


In [10]:
# Account for every input field as retained or excluded.
suffix_mapping = suffix_map(search_fields)
retained_suffix_fields = set(restore_suffix_names(suffix_mapping))
distinct_fields = set(search_fields)
excluded_suffix_fields = sorted(distinct_fields - retained_suffix_fields)
assert retained_suffix_fields <= distinct_fields
assert retained_suffix_fields | set(excluded_suffix_fields) == distinct_fields
assert retained_suffix_fields.isdisjoint(excluded_suffix_fields)
print(f"Shared suffix groups: {len(suffix_mapping)}")
print(f"Retained fields: {len(retained_suffix_fields)}; excluded fields: {len(excluded_suffix_fields)}")

Shared suffix groups: 48
Retained fields: 1283; excluded fields: 57


In [11]:
# Display shared suffixes and their remaining prefixes.
pprint(suffix_mapping, sort_dicts=False, width=100)

[{'buyer': ['acquiring-cpb',
            'concession-revenue',
            'concession-revenue-cur',
            'organisation-city',
            'organisation-contact-point',
            'organisation-country',
            'organisation-country-sub',
            'organisation-email',
            'organisation-fax',
            'organisation-gateway',
            'organisation-identifier',
            'organisation-internet-address',
            'organisation-name',
            'organisation-partname',
            'organisation-person',
            'organisation-post-code',
            'organisation-street',
            'organisation-tel',
            'touchpoint-city',
            'touchpoint-contact-point',
            'touchpoint-country',
            'touchpoint-country-sub',
            'touchpoint-email',
            'touchpoint-fax',
            'touchpoint-gateway',
            'touchpoint-internet-address',
            'touchpoint-name',
            'touchpoint-partname',
    

In [12]:
# Display fields whose last-token groups contain only one distinct field.
pprint(excluded_suffix_fields, width=100)

['AC',
 'FT',
 'HA',
 'NL',
 'RN',
 'RP',
 'TD',
 'TY',
 'activity-sector',
 'additional-information',
 'awarding-cpb-org',
 'buyer-category',
 'buyer-contracting-entity',
 'buyer-country-i18ns',
 'buyer-group-lead',
 'buyer-profile',
 'change-procurement-documents',
 'classification-cpv',
 'contract-framework-agreement',
 'contract-nature',
 'corporate-body',
 'country-origin',
 'deadline',
 'deadline-receipt-request',
 'direct-award-justification-previous-procedure-identifier-proct',
 'dispatch-time',
 'exclusion-grounds',
 'first-name-ubo',
 'funding',
 'kilometer-public-transport',
 'legal-basis',
 'legal-basis-no-id-desc',
 'legal-basis-text',
 'notice-purpose',
 'notice-version',
 'place-of-performance',
 'procedure-accelerated',
 'procedure-features',
 'procurement-relaunch',
 'registration-place-postcode',
 'registration-place-town',
 'review-request-decision',
 'review-request-fee',
 'review-request-withdrawn',
 'review-request-withdrawn-reasons',
 'selection-criteria-source',

## Referential Integrity

### Busines Codes -> Business Terms

### Business codes to business terms: equality join

Compare BT identifiers extracted from `codelists.csv.used_in` (left) with `business_terms.csv.business_term_identifier` (right). Repeated references are retained. This measures identifier-column join cardinality, rather than distinct codelist-to-term connections.

Pandas checks uniqueness across each complete input to select `one_to_one`, `one_to_many`, `many_to_one`, or `many_to_many`. An outer `merge` uses that `validate` setting and `indicator=True`: `both` means matched, `left_only` means an unknown referenced BT, and `right_only` means a business term with no codelist reference. Many-to-many permits duplicates without additional validation. 

The first row counts nonblank input keys, including duplicates. The second row identifies each input key; the third counts null and blank input values excluded from the join. Join rows counts the left join result for the first table and the right join result for the second table, including duplicate match combinations and unmatched rows on the retained side. All other counts refer to distinct keys. Common values occur in both fields. Matching once/many counts keys matching exactly one/more than one occurrence in the other field. Never matching counts keys with zero matches. Matching once and matching many sum to common values; never matching plus common values equals distinct values. Nulls and blank strings are excluded before merging so they cannot match each other. Leading zeros and suffixes such as `BT-67(a)` are preserved; OPP/OPT references are excluded. `bt_merge` retains the detailed join for inspection. This section requires pandas; the hierarchy sections use only the standard library.


In [4]:
import pandas as pd
import re


# Classify and validate an equality join, returning its summary and detailed matches.
def relationship_summary(left_values, right_values, left_name="left", right_name="right",
                         left_key="key", right_key="key"):
    def key_frame(values):
        keys = pd.Series(values, dtype="string").str.strip()
        missing = keys.isna() | keys.eq("")
        return keys[~missing].to_frame("business_term_identifier"), int(missing.sum())

    left, left_nulls = key_frame(left_values)
    right, right_nulls = key_frame(right_values)
    key = "business_term_identifier"
    relationships = {
        (True, True): "one_to_one",
        (True, False): "one_to_many",
        (False, True): "many_to_one",
        (False, False): "many_to_many",
    }
    validation = relationships[left[key].is_unique, right[key].is_unique]
    merged = left.merge(right, on=key, how="outer", validate=validation, indicator=True)

    # Count distinct keys and their matching occurrences in the other field.
    common = merged.loc[merged["_merge"].eq("both"), key].drop_duplicates()

    def checks_for(values, other_values, exclusive_label, null_count, key_name):
        match_counts = other_values[key].value_counts().reindex(common)
        exclusive = merged.loc[merged["_merge"].eq(exclusive_label), key]
        return [len(values), key_name, null_count, values[key].nunique(),
                len(common), exclusive.nunique(),
                int(match_counts.eq(1).sum()), int(match_counts.gt(1).sum()),
                int(merged["_merge"].isin(["both", exclusive_label]).sum())]

    summary = pd.DataFrame({
        "check": [
            "total rows",
            "key name",
            "null values",
            "distinct keys",
            "common keys",
            "keys never matching",
            "keys matching once",
            "keys matching many",
            "SIDE JOIN ROWS",
        ],
        left_name: checks_for(left, right, "left_only", left_nulls, left_key),
        right_name: checks_for(right, left, "right_only", right_nulls, right_key),
    })
    return summary, merged


In [5]:
# Load the two source fields relative to the existing search-field input.
semantic_scrape = input_path.parent.parent / "field semantics" / "web scrape"
with (semantic_scrape / "codelists.csv").open(encoding="utf-8-sig", newline="") as handle:
    codelist_rows = list(csv.DictReader(handle))
with (semantic_scrape / "business_terms.csv").open(encoding="utf-8-sig", newline="") as handle:
    business_term_rows = list(csv.DictReader(handle))

# Extract each BT reference, retaining repeated references across codelists.
bt_pattern = r"\bBT-\d+(?:\([a-zA-Z]+\))?(?![\w(-])"
codelist_bts = [
    bt
    for row in codelist_rows
    for bt in re.findall(bt_pattern, row["used_in"])
]
business_term_bts = [row["business_term_identifier"].strip() for row in business_term_rows]

bt_relationship_business, bt_merge_business = relationship_summary(
    codelist_bts, business_term_bts,
    left_name="codelists", left_key="used_in",
    right_name="business_terms", right_key="business_term_identifier",
)


display(bt_relationship_business)


,check,codelists,business_terms
0,rows,284,387
1,key name,used_in,business_term_identifier
2,null values,0,0
3,distinct keys,195,387
4,common keys,195,195
5,keys never matching,0,192
6,keys matching once,195,192
7,keys matching many,0,3
8,SIDE JOIN ROWS,284,476


In [10]:
# Read search-field records using the CSV path resolved above.
with input_path.open(encoding="utf-8-sig", newline="") as handle:
    search_field_list_rows = list(csv.DictReader(handle))

search_field_list = [row["Reference in the eForms SDK"].strip() for row in search_field_list_rows]

semantic_download = input_path.parent.parent / "field semantics" / "CSVs download"
with (semantic_download / "fields.csv").open(encoding="utf-8-sig", newline="") as handle:
    field_list_rows = list(csv.DictReader(handle))

bs_field_list = [row["fieldId"].strip() for row in field_list_rows]

fields_relationship, fields_merge = relationship_summary(
    search_field_list, bs_field_list,
    left_name="List_of_search_fields", left_key="Reference in the eForms SDK",
    right_name="fields", right_key="fieldId",
)

display(fields_relationship)

,check,List_of_search_fields,fields
0,rows,1327,1256
1,key name,Reference in the eForms SDK,fieldId
2,null values,13,0
3,distinct keys,544,1256
4,common keys,474,474
5,keys never matching,70,782
6,keys matching once,474,423
7,keys matching many,0,51
8,SIDE JOIN ROWS,1327,1983


In [31]:
import sys

# Import the shared builder from the semantic-base folder, regardless of notebook cwd.
semantic_base_folder = str(input_path.parent.parent)
if semantic_base_folder not in sys.path:
    sys.path.insert(0, semantic_base_folder)
from semantic_base_builder import build_semantic_base

# Load the additional semantic tables used for exploration.
bs_terms_df = pd.read_csv(
    semantic_scrape / "business_terms.csv",
    usecols=[
        "business_group_name",
        "business_term_identifier",
        "Description",
        "Data Type",
    ],
    encoding="utf-8-sig",
)

bs_codelists_df = pd.read_csv(
    semantic_scrape / "codelists.csv",
    usecols=[
        "codelist_id",
        "category",
        "description",
        "used_in"
    ],
    encoding="utf-8-sig",
)

bs_codelist_codes_df = pd.read_csv(
    semantic_scrape / "codelist_codes.csv",
    usecols=[
        "codelist_id",
        "code",
        "label",
    ],
    encoding="utf-8-sig",
)


In [ ]:
# Reuse the builder's normalization, fan-out, and validated left join.
api_search_fields_joined = build_semantic_base()

# Inspect all unmatched rows, including nonblank references.
api_search_fields_joined[
    ~api_search_fields_joined["sdk_field_key"].isin(
        pd.Series(bs_field_list, dtype="string").str.strip().str.lower()
    )
]


AttributeError: 'str' object has no attribute 'isnull'

In [30]:
from IPython.display import HTML

code_dictionary = bs_codelists_df.merge(
    bs_codelist_codes_df,
    on="codelist_id",
    how="left",
)

print(
"Rows:\n", code_dictionary.count(),
"\n"
)

display(HTML(
    f'<div style="max-height: 500px; overflow: auto;">'
    f'{code_dictionary.to_html(index=False)}'
    '</div>'
))

Rows:
 codelist_id    16911
category       16911
description    16911
used_in        12661
code           16911
label          16911
dtype: int64 



codelist_id,category,description,used_in,code,label
accessibility,Official Codelists,This table provides a list of options for the use of accessibility criteria for person with disabilities in the technical specifications within the domain of public procurement.,BT-754 Accessibility,inc,Accessibility criteria for persons with disabilities are included
accessibility,Official Codelists,This table provides a list of options for the use of accessibility criteria for person with disabilities in the technical specifications within the domain of public procurement.,BT-754 Accessibility,n-inc,Accessibility criteria for persons with disabilities are not included because the procurement is not intended for use by natural persons
accessibility,Official Codelists,This table provides a list of options for the use of accessibility criteria for person with disabilities in the technical specifications within the domain of public procurement.,BT-754 Accessibility,n-inc-just,Accessibility criteria for persons with disabilities are not included with the following justification
applicability,Official Codelists,This table provides a list of the options pertinent to be chosen for a matter concerning the applicability of predefined fields. It is designed for but not restricted to the context of public procurement.,BT-773 Subcontracting,no,No
applicability,Official Codelists,This table provides a list of the options pertinent to be chosen for a matter concerning the applicability of predefined fields. It is designed for but not restricted to the context of public procurement.,BT-773 Subcontracting,not-known,Not yet known
applicability,Official Codelists,This table provides a list of the options pertinent to be chosen for a matter concerning the applicability of predefined fields. It is designed for but not restricted to the context of public procurement.,BT-773 Subcontracting,yes,Yes
award-criterion-type,Official Codelists,List of award criterion codes for award decisions within public procurement procedures.,BT-539 Award Criterion Type,cost,Cost
award-criterion-type,Official Codelists,List of award criterion codes for award decisions within public procurement procedures.,BT-539 Award Criterion Type,price,Price
award-criterion-type,Official Codelists,List of award criterion codes for award decisions within public procurement procedures.,BT-539 Award Criterion Type,quality,Quality
buyer-legal-type,Official Codelists,This table provides a list of the the type of procuring authorities within the domain of public procurement according to legislation.,BT-11 Buyer Legal Type,body-pl,Body governed by public law
